# 03 — MILP version 1: choosing the tracks

**Problem:** for a given moment, choose among the candidates a set of tracks that fills the time and that I listen to as much as possible. The order does not matter (notebook 04).

- Variables: $x_i \in \{0,1\}$, 1 if track $i$ enters the playlist.
- Objective: $\max \sum_i (v_i - \lambda e_i)\, x_i$, with $v_i = (1 - (1-f)\,p_i)\,d_i$ = expected minutes listened and $e_i = \max(0, d_i - d_{ref})$ = minutes beyond a normal length.
- Constraints:
  - duration: $T_{min} \le \sum_i d_i x_i \le T_{max}$;
  - variety: $\sum_{i \in I_a} x_i \le K$ for every artist $a$;
  - new tracks: $q_{min} \sum_i x_i \le \sum_{i \in N} x_i \le q_{max} \sum_i x_i$ (share, linearized by multiplying by the denominator);
  - long tracks: $\sum_i e_i x_i \le s \sum_i d_i x_i$.
- $p_i$ = probability of skipping track $i$ after a non-skipped track (`p0` of the model, `src/train.py`); $d_i$ = estimated duration (`src/candidates.py`); $f$ = mean fraction of a track listened when I skip it.

The model is in `src/playlist.py`. Moment of the example: Monday 26/10/2026 at 8:00, phone, Germany, shuffle.

In [1]:
import sys
import time

import joblib
import numpy as np
import pandas as pd
import pulp

sys.path.append("../src")
from features import ROOT
from playlist import F, LAMBDA_EXCESS, PARAMS, add_day_constraints, solve_v1
from train import MODEL_PATH, candidate_features, predict_candidates

streams = pd.read_parquet(ROOT / "data" / "streams_clean.parquet")
pool = pd.read_parquet(ROOT / "data" / "pool.parquet")
model = joblib.load(MODEL_PATH)

# History features at the end of the history (20/10), context of the playlist moment.
# shuffle=True for every track: it shifts the level of the probabilities, it hardly changes
# the ranking between tracks.
WHEN = "2026-10-26 08:00"
t0 = time.perf_counter()
context = dict(platform="ios", country="DE", shuffle=True)
feats = candidate_features(streams, pool["track_uri"], WHEN, **context)
cand = predict_candidates(model, feats)
cand = cand.merge(pool[["track_uri", "duration_s", "is_new", "plays"]], on="track_uri")
cand["d"] = cand["duration_s"] / 60  # duration in minutes
print(f"{len(cand)} candidates, predictions in {time.perf_counter() - t0:.1f} s")
print(cand.groupby("is_new")["p0"].describe().round(3))

4323 candidates, predictions in 0.1 s
         count   mean    std    min    25%    50%    75%    max
is_new                                                         
False   3689.0  0.084  0.042  0.040  0.056  0.073  0.097  0.370
True     634.0  0.060  0.022  0.042  0.048  0.051  0.063  0.204


## Value of each track

In [2]:
# f: fraction of the track listened before a forward skip, on the candidates
dur = pool.set_index("track_uri")["duration_s"]
fwd = streams[(streams["reason_end"] == "fwdbtn") & streams["track_uri"].isin(dur.index)]
frac = (fwd["ms_played"] / 1000 / fwd["track_uri"].map(dur)).clip(upper=1)
recent = fwd["ts"] >= pd.Timestamp("2025-07-01", tz="UTC")
print(
    f"fraction listened when skipping: mean {frac.mean():.2f}, median {frac.median():.2f}, "
    f"mean since 2025-07 {frac[recent].mean():.2f}"
)
print(f"f used (F in src/playlist.py): {F}")

cand["v"] = (1 - (1 - F) * cand["p0"]) * cand["d"]  # expected minutes listened if included

fraction listened when skipping: mean 0.21, median 0.02, mean since 2025-07 0.22
f used (F in src/playlist.py): 0.2


## Solution

In [3]:
res = solve_v1(cand, **PARAMS)
print(
    f"{res['status']}: {res['obj']:.2f} expected minutes, {res['n_constraints']} constraints, "
    f"{res['secs']:.2f} s"
)


def summary(sel):
    """Summary of a playlist (selected rows of cand)."""
    return {
        "tracks": len(sel), "artists": sel["artist"].nunique(), "duration": sel["d"].sum(),
        "expected minutes": sel["v"].sum(), "mean p0": sel["p0"].mean(),
        "new share": sel["is_new"].mean(),
    }


# Titles and artists stay out of the notebook: python src/playlist.py shows them in the terminal
pl = cand[res["x"] > 0.5]
print(pd.Series(summary(pl)).round(3), "\n")
print(pl.sort_values("p0")[["d", "p0", "is_new", "plays"]].round(3).to_string(index=False))

Optimal: 43.51 expected minutes, 283 constraints, 0.72 s
tracks              15.000
artists              9.000
duration            44.998
expected minutes    43.515
mean p0              0.041
new share            0.133
dtype: float64 

    d    p0  is_new  plays
3.952 0.040   False     11
3.950 0.040   False     20
2.796 0.041   False     13
3.159 0.041   False     21
3.102 0.041   False      5
1.748 0.041   False      8
2.634 0.041   False     28
2.953 0.041   False      8
3.167 0.042   False      6
3.679 0.042   False      6
2.898 0.042   False     16
3.962 0.042   False      4
3.921 0.042   False      5
1.622 0.043    True      1
1.454 0.043    True      1


## Long tracks: from a threshold to a penalty

Four versions of the rule on long tracks, for playlists of 30, 45 and 120 minutes. "Expected minutes" without the penalty term, so that the versions can be compared.

In [4]:
def solve_threshold(cand, params, d_long=5, share=0.2):
    """Version 1 with a sharp threshold instead of the excess: minutes on tracks longer than
    d_long at most a share of the total.
    """
    I = range(len(cand))
    d, v = cand["d"].to_numpy(), cand["v"].to_numpy()
    prob = pulp.LpProblem("threshold", pulp.LpMaximize)
    x = [prob.add_variable(f"x_{i}", lowBound=0, upBound=1, cat="Binary") for i in I]
    prob += pulp.lpSum(v[i] * x[i] for i in I)
    add_day_constraints(prob, x, cand, **dict(params, s_excess=100))  # excess constraint off
    total = pulp.lpSum(d[i] * x[i] for i in I)
    prob += pulp.lpSum(d[i] * x[i] for i in I if d[i] > d_long) <= share * total, "long_share"
    prob.solve(pulp.HiGHS(msg=False))
    return np.array([x[i].value() for i in I])


versions = {
    "1. no rule": lambda p: solve_v1(cand, **dict(p, s_excess=100), lam_excess=0)["x"],
    "2. sharp threshold": lambda p: solve_threshold(cand, p),
    "3. excess constraint": lambda p: solve_v1(cand, **p, lam_excess=0)["x"],
    "4. constraint + penalty": lambda p: solve_v1(cand, **p)["x"],
}
rows = []
for name, solve in versions.items():
    for minutes in [30, 45, 120]:
        sel = cand[solve(dict(PARAMS, T_min=minutes - 5, T_max=minutes)) > 0.5]
        rows.append({
            "version": name, "minutes": minutes, "tracks": len(sel), "longest": sel["d"].max(),
            "over 4 min": (sel["d"] > 4).sum(), "4.5-5 min": sel["d"].between(4.5, 5).sum(),
            "expected minutes": sel["v"].sum(),
        })
pd.DataFrame(rows).round(3)

,version,minutes,tracks,longest,over 4 min,4.5-5 min,expected minutes
0,1. no rule,30,9,7.661,2,0,29.025
1,1. no rule,45,13,7.661,3,0,43.523
2,1. no rule,120,36,7.661,7,0,115.945
3,2. sharp threshold,30,10,4.975,2,2,29.020
4,2. sharp threshold,45,14,7.661,2,1,43.524
5,2. sharp threshold,120,37,7.661,7,0,115.943
6,3. excess constraint,30,10,4.275,1,0,29.019
7,3. excess constraint,45,14,5.029,2,1,43.517
8,3. excess constraint,120,37,7.661,5,0,115.934
9,4. constraint + penalty,30,10,3.952,0,0,29.018


## Greedy heuristic

Simple rule: sort the tracks by $v_i/d_i$ (expected minutes per minute of duration, i.e. increasing $p_i$) and add them while the constraints allow it. If the MILP optimum is almost the same, version 1 does not need a solver: a result worth knowing, not a failure.

In [5]:
def greedy_v1(cand, T_min, T_max, K, q_min, q_max, d_ref, s_excess):
    """Greedy rule: tracks by decreasing v/d, added while the constraints allow it."""
    order = (cand["v"] / cand["d"]).sort_values(ascending=False).index
    chosen, per_artist, dur, n_new, exc = [], {}, 0.0, 0, 0.0
    for i in order:
        r = cand.loc[i]
        if dur + r["d"] > T_max or per_artist.get(r["artist"], 0) >= K:
            continue
        if r["is_new"] and n_new + 1 > q_max * (len(chosen) + 1):
            continue
        if exc + max(r["d"] - d_ref, 0) > s_excess * T_min:
            continue
        chosen.append(i)
        per_artist[r["artist"]] = per_artist.get(r["artist"], 0) + 1
        dur += r["d"]
        n_new += int(r["is_new"])
        exc += max(r["d"] - d_ref, 0)
        if dur >= T_min and T_max - dur < cand["d"].min():
            break
    return cand.loc[chosen]


gr = greedy_v1(cand, **PARAMS)
print(pd.DataFrame({"MILP": summary(pl), "greedy": summary(gr)}).round(3))
print(f"\nq_min respected by the greedy rule: {gr['is_new'].mean() >= PARAMS['q_min']}")
print(f"Tracks in common: {len(set(pl.index) & set(gr.index))} of {len(pl)}")

                    MILP  greedy
tracks            15.000  16.000
artists            9.000  10.000
duration          44.998  44.940
expected minutes  43.515  43.457
mean p0            0.041   0.041
new share          0.133   0.000

q_min respected by the greedy rule: False
Tracks in common: 9 of 15


## Random playlists

Reference: tracks taken at random from the candidates until the duration is filled (like a shuffle over the library). It measures how much the choice of the tracks is worth.

In [6]:
rng = np.random.default_rng(0)


def random_playlist(cand, T_max, rng):
    """Tracks in random order, added while the duration stays below T_max."""
    perm = cand.iloc[rng.permutation(len(cand))]
    return perm[perm["d"].cumsum() <= T_max]


rand = pd.DataFrame([summary(random_playlist(cand, PARAMS["T_max"], rng)) for _ in range(500)])
print(rand.mean().round(3))

# Random playlists fill the duration less: fair comparison through the expected fraction listened
share_milp = summary(pl)["expected minutes"] / summary(pl)["duration"]
share_rand = (rand["expected minutes"] / rand["duration"]).mean()
print(f"\nExpected fraction listened: MILP {share_milp:.3f}, random {share_rand:.3f}")
print(f"Mean p0: MILP {summary(pl)['mean p0']:.3f}, random {rand['mean p0'].mean():.3f}")

tracks              13.246
artists             12.726
duration            43.266
expected minutes    40.460
mean p0              0.081
new share            0.148
dtype: float64

Expected fraction listened: MILP 0.967, random 0.935
Mean p0: MILP 0.041, random 0.081


## LP relaxation

With continuous $x_i \in [0, 1]$ the optimum is an upper bound for the integer one. A small difference and few fractional values mean an easy problem for branch and bound.

In [7]:
lp = solve_v1(cand, **PARAMS, relax=True)
n_frac = ((lp["x"] > 1e-6) & (lp["x"] < 1 - 1e-6)).sum()
print(
    f"LP: {lp['obj']:.3f}   MILP: {res['obj']:.3f}   "
    f"difference: {lp['obj'] - res['obj']:.3f} minutes"
)
print(f"Fractional variables in the LP: {n_frac} of {len(cand)}")

LP: 43.519   MILP: 43.515   difference: 0.004 minutes
Fractional variables in the LP: 2 of 4323


## Sensitivity to the parameters

What the constraints cost: expected minutes lost with respect to the problem without variety and without the new-track share.

In [8]:
rows = []
for K in [1, 2, 3, 5, 1000]:
    for q_min in [0.0, 0.1, 0.3]:
        p = dict(PARAMS, K=K, q_min=q_min, q_max=1.0 if q_min == 0 else PARAMS["q_max"])
        r = solve_v1(cand, **p)
        sel = cand[r["x"] > 0.5]
        row = {"K": K, "q_min": q_min, "q_max": p["q_max"], **summary(sel), "secs": r["secs"]}
        rows.append(row)
pd.DataFrame(rows).round(3)

,K,q_min,q_max,tracks,artists,duration,expected minutes,mean p0,new share,secs
0,1,0.0,1.0,15,15,44.997,43.494,0.042,0.133,0.519
1,1,0.1,0.4,15,15,44.999,43.495,0.042,0.133,0.489
2,1,0.3,0.4,16,16,44.999,43.489,0.042,0.312,0.726
3,2,0.0,1.0,15,9,44.998,43.517,0.041,0.000,0.612
4,2,0.1,0.4,15,9,44.998,43.515,0.041,0.133,0.695
5,2,0.3,0.4,16,11,45.000,43.507,0.042,0.312,0.658
6,3,0.0,1.0,14,8,44.999,43.528,0.041,0.000,0.221
7,3,0.1,0.4,15,9,45.000,43.524,0.041,0.133,0.377
8,3,0.3,0.4,16,8,44.999,43.517,0.041,0.312,0.644
9,5,0.0,1.0,15,6,44.999,43.541,0.041,0.000,0.512


In [9]:
# How many tracks have a skip probability close to the minimum?
p_min = cand["p0"].min()
for eps in [0.001, 0.002, 0.005, 0.01]:
    near = cand["p0"] <= p_min + eps
    print(
        f"p0 within {eps:.3f} of the minimum ({p_min:.3f}): "
        f"{near.sum()} tracks, {cand.loc[near, 'd'].sum():.0f} minutes of music"
    )

p0 within 0.001 of the minimum (0.040): 38 tracks, 152 minutes of music
p0 within 0.002 of the minimum (0.040): 54 tracks, 202 minutes of music
p0 within 0.005 of the minimum (0.040): 185 tracks, 606 minutes of music
p0 within 0.010 of the minimum (0.040): 755 tracks, 2467 minutes of music


## Rotation over several days

A track should not come back too soon. Two ways to plan the 7 days from Monday 26/10 (8:00), with a track at most once every $k$ days:
- **joint MILP:** variables $x_{i,g}$ for track $i$ and day $g$, the constraints of a playlist for every day, and $\sum_{h=g}^{g+k-1} x_{i,h} \le 1$ for every window of $k$ days;
- **daily filter:** every day version 1, after removing the tracks used in the previous $k-1$ days (as `daily_playlist` in `src/playlist.py`).

In both, a track played $r$ days before a plan day is excluded if $r < k$.

In [10]:
DAYS = pd.date_range(WHEN, periods=7, freq="D", tz="Europe/Berlin")


def day_candidates(when):
    """Candidates with their value for the playlist of `when` (same columns as cand)."""
    c = predict_candidates(model, candidate_features(streams, pool["track_uri"], when, **context))
    c = c.merge(pool[["track_uri", "duration_s", "is_new"]], on="track_uri")
    c["d"] = c["duration_s"] / 60
    c["v"] = (1 - (1 - F) * c["p0"]) * c["d"]
    return c


day_cand = [day_candidates(w) for w in DAYS]
n, D = len(cand), len(DAYS)
assert all((c["track_uri"] == day_cand[0]["track_uri"]).all() for c in day_cand)
print("mean p0 per day:", " ".join(f"{c['p0'].mean():.3f}" for c in day_cand))
corr = np.corrcoef(day_cand[0]["p0"], day_cand[5]["p0"])[0, 1]
print(f"correlation of p0, Monday vs Saturday: {corr:.4f}")

# Days between the last play of every candidate and the first day of the plan
last_play = streams.groupby("track_uri")["ts"].max().dt.tz_convert("Europe/Berlin").dt.normalize()
days_before = (DAYS[0].normalize() - day_cand[0]["track_uri"].map(last_play)).dt.days.to_numpy()


def expected_minutes(X, days_cand):
    """Expected minutes of a plan X (days x tracks, True = in the playlist of that day)."""
    return sum(c["v"].to_numpy()[X[g]].sum() for g, c in enumerate(days_cand))


def solve_joint(k, **solver):
    """Joint MILP over the days: one playlist per day, a track at most once every k days."""
    allowed = days_before[:, None] + np.arange(D)[None, :] >= k  # history: r >= k
    prob = pulp.LpProblem("rotation", pulp.LpMaximize)
    # Arguments of add_variable: name, lower bound, upper bound, type
    x = [
        [prob.add_variable(f"x_{i}_{g}", 0, int(allowed[i, g]), "Binary") for i in range(n)]
        for g in range(D)
    ]
    obj = []
    for g, c in enumerate(day_cand):
        v, e = c["v"].to_numpy(), np.maximum(c["d"].to_numpy() - PARAMS["d_ref"], 0)
        obj += [(v[i] - LAMBDA_EXCESS * e[i]) * x[g][i] for i in range(n)]
        add_day_constraints(prob, x[g], c, **PARAMS, tag=f"_g{g}")  # unique constraint names
    prob += pulp.lpSum(obj)
    for i in range(n):
        for g in range(D - k + 1):
            prob += pulp.lpSum(x[h][i] for h in range(g, g + k)) <= 1, f"rotation_{i}_{g}"
    stats = prob.solve(pulp.HiGHS(msg=False, **solver))
    X = np.array([[x[g][i].value() for i in range(n)] for g in range(D)]) > 0.5
    return stats, X


def solve_filter(k, days_cand):
    """Every day version 1 without the tracks used or played in the previous k - 1 days."""
    last = -days_before.copy()  # day of the last use (negative: before the plan)
    X = np.zeros((len(days_cand), n), dtype=bool)
    for g, c in enumerate(days_cand):
        ok = g - last >= k
        r = solve_v1(c[ok], **PARAMS)
        chosen = np.flatnonzero(ok)[r["x"] > 0.5]
        X[g, chosen] = True
        last[chosen] = g
    return X

mean p0 per day: 0.081 0.081 0.081 0.081 0.081 0.092 0.092
correlation of p0, Monday vs Saturday: 0.9999


In [11]:
rows = []
for k in [3, 7]:
    t0 = time.perf_counter()
    X = solve_filter(k, day_cand)
    rows.append({
        "k": k, "method": "daily filter", "upper bound": np.nan,
        "expected minutes": expected_minutes(X, day_cand),
        "distinct tracks": X.any(axis=0).sum(), "secs": time.perf_counter() - t0,
    })
    # Joint model stopped at a relative gap of 0.5% (the default, 0.01%, takes much longer)
    t0 = time.perf_counter()
    stats, X = solve_joint(k, gapRel=0.005, timeLimit=120)
    rows.append({
        "k": k, "method": "joint, gap 0.5%", "upper bound": stats.best_bound,
        "expected minutes": expected_minutes(X, day_cand),
        "distinct tracks": X.any(axis=0).sum(), "secs": time.perf_counter() - t0,
    })
pd.DataFrame(rows).round(2)

,k,method,upper bound,expected minutes,distinct tracks,secs
0,3,daily filter,NaN,304.05,53,4.51
1,3,"joint, gap 0.5%",304.07,302.79,58,15.75
2,7,daily filter,NaN,303.84,109,3.52
3,7,"joint, gap 0.5%",303.88,302.38,106,16.21


In [12]:
# Joint model with the default gap: how far does branch and bound get in 30 seconds?
stats, X = solve_joint(3, timeLimit=30)
print(
    f"{stats.status_str}: {expected_minutes(X, day_cand):.2f} expected minutes, "
    f"upper bound {stats.best_bound:.2f}, gap {stats.gap_rel:.2%}"
)

TimeLimit: 303.54 expected minutes, upper bound 304.07, gap 0.18%


In [13]:
# Length of the rotation for the experiment: 4 weeks of daily playlists with the filter.
# During the experiment the history does not update, so the model does not see the wear of
# repeated tracks: only the rotation limits it.
DAYS_28 = pd.date_range(WHEN, periods=28, freq="D", tz="Europe/Berlin")
cand_28 = [day_candidates(w) for w in DAYS_28]
rows = []
for k in [3, 7, 14]:
    X = solve_filter(k, cand_28)
    uses = X.sum(axis=0)
    p0_chosen = np.concatenate([c["p0"].to_numpy()[X[g]] for g, c in enumerate(cand_28)])
    rows.append({
        "k": k, "expected minutes per day": expected_minutes(X, cand_28) / len(cand_28),
        "mean p0": p0_chosen.mean(), "max p0": p0_chosen.max(),
        "distinct tracks": (uses > 0).sum(), "max repetitions": uses.max(),
    })
pd.DataFrame(rows).round(4)

,k,expected minutes per day,mean p0,max p0,distinct tracks,max repetitions
0,3,43.4341,0.0436,0.0524,63,10
1,7,43.4065,0.0444,0.0524,111,4
2,14,43.3712,0.0453,0.0535,224,2


## Conclusions

- **Solution:** 15 tracks, 45.0 minutes, 43.51 expected minutes, mean `p0` 0.041, 11 artists, 20% new tracks. HiGHS solves it in about 0.5 s with 283 constraints (the artist constraint is written only for the artists with more than K candidates).
- **f = 0.2:** before a forward skip I listen to 21% of the track on average (median 2%: half of the skips are immediate; 22% since 2025-07).
- **Long tracks, a modelling lesson.**
  1. Without a rule the solver takes tracks of 7.7 minutes. The value per minute depends only on $p_i$, so among almost tied tracks it takes the ones that fill the duration exactly: a preference that is not written in the model does not exist for the solver.
  2. A sharp threshold (minutes on tracks over 5 minutes at most 20% of the total) is respected but exploited: it still lets the 7.7-minute track in and adds one just below 5 minutes. A sharp threshold creates a jump that the optimizer uses.
  3. The continuous excess constraint has no jump, but the solver uses all the margin it leaves (at 30 minutes a 5.45-minute track; at 2 hours five tracks over 4 minutes, one of 7.7).
  4. The penalty λ in the objective decides among the tied tracks: no track over 4 minutes at 30, 45 and 120 minutes, for 0.002 expected minutes at 45 minutes (0.015 at 2 hours). The constraint stays as a guarantee, the penalty expresses the preference.
- **Greedy rule:** 43.46 against 43.51 expected minutes, and it violates $q_{min}$ (no new track). The advantage of the MILP is that all the constraints hold together, not a better objective.
- **Random playlists:** mean `p0` 0.081 against 0.041, expected fraction listened 0.935 against 0.967. These are *predicted* values: the A/B test has to verify them.
- **LP relaxation:** 43.519 against 43.513, 2 fractional variables out of 4,323: an easy problem for branch and bound.
- **Cost of the constraints:** at most 0.07 expected minutes between no limit per artist (but then one artist fills the playlist) and K = 1.
- **Ties at the top:** the chosen tracks have `p0` between 0.040 and 0.043; 38 tracks are within 0.001 of the minimum (152 minutes of music), 185 within 0.005 (606 minutes). These differences are much smaller than the error of the model: among the good tracks the choice is practically random. Version 1 separates good tracks from bad ones (`p0` 0.041 against 0.081 at random); it does not rank the good ones. To be stated in the README.
- **Rotation over several days.** On weekdays `p0` is identical (the history features stop at the end of the history), the weekend only shifts the level (0.081 → 0.092). With k = 3 the daily filter reaches 304.05 expected minutes in the week, within 0.02 of the upper bound of the joint model (304.07), in 3.7 s; the joint model stopped at a 0.5% gap gives 303.25 in 15 s, and with the default gap it is still at 0.26% after 30 s. Same picture with k = 7. Probable cause: symmetry. The days are almost interchangeable, so swapping the playlists of two days gives a solution of the same value, and branch and bound cannot rule these out. **Decision:** no joint model; every day version 1 with the filter (`daily_playlist` in `src/playlist.py`).
- **Length of the rotation:** over 4 weeks, k = 3 repeats the best tracks up to 10 times (59 distinct tracks), k = 14 at most twice (225 tracks), for 0.06 expected minutes per day. During the experiment the model does not see the wear of repeated tracks, so `K_DAYS = 14`.